## Install dependencies

In [1]:
# Run this once if needed
!pip install yt-dlp youtube-transcript-api


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import libraries

In [2]:
import os
import json
import re
import time
from datetime import datetime
from urllib.parse import urlparse, parse_qs
from concurrent.futures import ThreadPoolExecutor, as_completed

from youtube_transcript_api import YouTubeTranscriptApi
from yt_dlp import YoutubeDL

## Helper functions

These helpers let you paste **one YouTube URL** and the notebook will detect whether it is a:
- normal YouTube video
- YouTube Shorts URL
- playlist URL

In [3]:
def detect_url_type(url: str) -> str:
    """
    Detect whether a YouTube URL is a video, short, or playlist.
    Priority: playlist first, because watch URLs may also contain list=...
    """
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()
    query = parse_qs(parsed.query)
    path = parsed.path or ""

    if "list" in query:
        return "playlist"

    if "/shorts/" in path:
        return "short"

    if host == "youtu.be":
        return "video"

    if path == "/watch" and "v" in query:
        return "video"

    return "unknown"


def get_video_id(url: str) -> str:
    """
    Extract YouTube video ID from common URL formats:
    - https://www.youtube.com/watch?v=VIDEO_ID
    - https://youtu.be/VIDEO_ID
    - https://www.youtube.com/embed/VIDEO_ID
    - https://www.youtube.com/shorts/VIDEO_ID
    Also tolerates extra query params like ?si=...&list=...
    """
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()

    if host == "youtu.be":
        vid = parsed.path.lstrip("/").split("/")[0]
        if vid and len(vid) == 11:
            return vid

    if host in {"www.youtube.com", "youtube.com", "m.youtube.com"}:
        if parsed.path == "/watch":
            q = parse_qs(parsed.query)
            vid = (q.get("v") or [None])[0]
            if vid and len(vid) == 11:
                return vid

        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] in {"embed", "shorts"}:
            vid = parts[1]
            if vid and len(vid) == 11:
                return vid

    match = re.search(r"([a-zA-Z0-9_-]{11})", url)
    if match:
        return match.group(1)

    raise ValueError(f"Could not extract a valid YouTube video ID from: {url}")

## Universal collector

This version supports:
- one normal YouTube video URL
- one YouTube Shorts URL
- one playlist URL
- skipping videos that already exist on disk
- sequential mode
- parallel mode for larger playlists

In [11]:
import json
import os
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

from yt_dlp import YoutubeDL
from youtube_transcript_api import YouTubeTranscriptApi


class UniversalYouTubeCollector:
    def __init__(self, output_dir="../data/raw", sleep_seconds=0.0):
        self.output_dir = output_dir
        self.sleep_seconds = sleep_seconds
        os.makedirs(self.output_dir, exist_ok=True)

    def _build_ydl_options(self, extract_flat=True):
        return {
            "quiet": True,
            "no_warnings": True,
            "skip_download": True,
            "extract_flat": extract_flat,
            "ignoreerrors": True,
        }

    def _video_output_path(self, video_id: str) -> str:
        return os.path.join(self.output_dir, f"video_{video_id}.json")

    def _build_metadata(self, entry: dict) -> dict:
        video_id = entry["video_id"]
        return {
            "video_id": video_id,
            "title": entry.get("title"),
            "description": entry.get("description"),
            "duration": entry.get("duration"),
            "upload_date": entry.get("upload_date"),
            "uploader": entry.get("uploader"),
            "channel": entry.get("channel") or entry.get("uploader"),
            "channel_id": entry.get("channel_id"),
            "view_count": entry.get("view_count"),
            "like_count": entry.get("like_count"),
            "comment_count": entry.get("comment_count"),
            "tags": entry.get("tags") or [],
            "categories": entry.get("categories") or [],
            "thumbnail": entry.get("thumbnail"),
            "url": entry.get("url") or f"https://www.youtube.com/watch?v={video_id}",
            "playlist_index": entry.get("playlist_index"),
        }

    def fetch_transcript(self, video_id: str, languages=("en",)) -> dict:
        try:
            api = YouTubeTranscriptApi()
            fetched = api.fetch(video_id, languages=languages)
            return {
                "transcript": fetched.to_raw_data(),
                "transcript_language": fetched.language_code,
                "transcript_status": "ok",
            }
        except Exception as e:
            return {
                "transcript": [],
                "transcript_language": None,
                "transcript_status": f"unavailable: {e}",
            }

    def save_video_data(self, video_data: dict) -> str:
        output_file = self._video_output_path(video_data["video_id"])
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(video_data, f, indent=2, ensure_ascii=False)
        return output_file

    def process_entry(self, entry: dict, languages=("en",), save=True, overwrite=False):
        video_id = entry["video_id"]
        output_file = self._video_output_path(video_id)
        metadata = self._build_metadata(entry)

        if os.path.exists(output_file) and not overwrite:
            print("Skipping existing video")
            return {
                "video_id": video_id,
                "metadata": metadata,
                "transcript": None,
                "transcript_language": None,
                "transcript_status": "skipped_existing",
                "collected_at": datetime.now().isoformat(),
                "saved_to": output_file,
            }

        transcript_info = self.fetch_transcript(video_id, languages=languages)

        video_data = {
            "video_id": video_id,
            "metadata": metadata,
            "transcript": transcript_info["transcript"],
            "transcript_language": transcript_info["transcript_language"],
            "transcript_status": transcript_info["transcript_status"],
            "collected_at": datetime.now().isoformat(),
        }

        if save:
            video_data["saved_to"] = self.save_video_data(video_data)

        if self.sleep_seconds > 0:
            time.sleep(self.sleep_seconds)

        return video_data

    def get_playlist_entries(self, playlist_url: str, max_videos: int = None) -> list:
        with YoutubeDL(self._build_ydl_options(extract_flat=True)) as ydl:
            info = ydl.extract_info(playlist_url, download=False)

        raw_entries = info.get("entries", []) or []
        entries = []

        for entry in raw_entries:
            if not entry:
                continue

            video_id = entry.get("id")
            if not video_id:
                url = entry.get("url")
                if url:
                    try:
                        video_id = get_video_id(url)
                    except Exception:
                        continue

            if not video_id:
                continue

            entries.append({
                "video_id": video_id,
                "url": f"https://www.youtube.com/watch?v={video_id}",
                "title": entry.get("title"),
                "description": entry.get("description"),
                "duration": entry.get("duration"),
                "upload_date": entry.get("upload_date"),
                "uploader": entry.get("uploader"),
                "channel": entry.get("channel") or entry.get("uploader"),
                "channel_id": entry.get("channel_id"),
                "view_count": entry.get("view_count"),
                "like_count": entry.get("like_count"),
                "comment_count": entry.get("comment_count"),
                "tags": entry.get("tags") or [],
                "categories": entry.get("categories") or [],
                "thumbnail": entry.get("thumbnail"),
                "playlist_index": entry.get("playlist_index"),
            })

            if max_videos is not None and len(entries) >= max_videos:
                break

        print(f"Found {len(entries)} videos in playlist")
        return entries

    def get_single_video_entry(self, video_url: str) -> dict:
        with YoutubeDL(self._build_ydl_options(extract_flat=False)) as ydl:
            info = ydl.extract_info(video_url, download=False)

        video_id = info.get("id") or get_video_id(video_url)

        return {
            "video_id": video_id,
            "url": f"https://www.youtube.com/watch?v={video_id}",
            "title": info.get("title"),
            "description": info.get("description"),
            "duration": info.get("duration"),
            "upload_date": info.get("upload_date"),
            "uploader": info.get("uploader"),
            "channel": info.get("channel") or info.get("uploader"),
            "channel_id": info.get("channel_id"),
            "view_count": info.get("view_count"),
            "like_count": info.get("like_count"),
            "comment_count": info.get("comment_count"),
            "tags": info.get("tags") or [],
            "categories": info.get("categories") or [],
            "thumbnail": info.get("thumbnail"),
            "playlist_index": None,
        }

    def collect_single_video(self, video_url: str, languages=("en",), overwrite=False) -> dict:
        entry = self.get_single_video_entry(video_url)

        print(f"Collecting data for video: {entry['video_id']}")
        print(f"Title: {entry.get('title')}")
        print(f"Duration: {entry.get('duration')} seconds")

        result = self.process_entry(entry, languages=languages, save=True, overwrite=overwrite)

        if result.get("transcript_status") == "skipped_existing":
            print(f"Saved to: {result.get('saved_to')}")
            print("Video already exists. Transcript was not fetched again.")
        else:
            transcript_count = len(result.get("transcript") or [])
            transcript_lang = result.get("transcript_language")
            print(f"Transcript segments: {transcript_count} lang: {transcript_lang}")
            print(f"Saved to: {result.get('saved_to')}")

        return result

    def collect_playlist(self, playlist_url: str, max_videos: int = None, languages=("en",), overwrite=False) -> list:
        entries = self.get_playlist_entries(playlist_url, max_videos=max_videos)
        results = []

        for i, entry in enumerate(entries, 1):
            print(f"Processing {i}/{len(entries)}: {entry.get('title')}")
            try:
                result = self.process_entry(entry, languages=languages, save=True, overwrite=overwrite)
                results.append(result)
            except Exception as e:
                print(f"Failed for {entry['url']}: {e}")

        return results

    def collect_playlist_parallel(self, playlist_url: str, max_videos: int = None,
                                  languages=("en",), max_workers=4, overwrite=False) -> list:
        entries = self.get_playlist_entries(playlist_url, max_videos=max_videos)
        results = []

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(self.process_entry, entry, languages, True, overwrite): entry
                for entry in entries
            }

            for idx, future in enumerate(as_completed(futures), 1):
                entry = futures[future]
                try:
                    result = future.result()
                    print(f"Completed {idx}/{len(entries)}: {entry.get('title')}")
                    results.append(result)
                except Exception as e:
                    print(f"Failed for {entry['url']}: {e}")

        results.sort(key=lambda x: (
            x["metadata"].get("playlist_index") is None,
            x["metadata"].get("playlist_index") or 0
        ))
        return results

    def collect_from_url(self, youtube_url: str, max_videos: int = None,
                         languages=("en",), parallel=True, max_workers=4, overwrite=False) -> list:
        url_type = detect_url_type(youtube_url)
        print(f"Detected URL type: {url_type}")

        if url_type == "playlist":
            if parallel:
                return self.collect_playlist_parallel(
                    playlist_url=youtube_url,
                    max_videos=max_videos,
                    languages=languages,
                    max_workers=max_workers,
                    overwrite=overwrite,
                )
            return self.collect_playlist(
                playlist_url=youtube_url,
                max_videos=max_videos,
                languages=languages,
                overwrite=overwrite,
            )

        if url_type in {"video", "short"}:
            result = self.collect_single_video(
                youtube_url,
                languages=languages,
                overwrite=overwrite,
            )
            return [result]

        raise ValueError("Unsupported YouTube URL. Please provide a valid video, short, or playlist URL.")

    def save_playlist_summary(self, results: list, filename="playlist_summary.json"):
        output_file = os.path.join(self.output_dir, filename)
        summary = {
            "total_videos": len(results),
            "generated_at": datetime.now().isoformat(),
            "videos": [
                {
                    "video_id": r["video_id"],
                    "title": r["metadata"].get("title"),
                    "url": r["metadata"].get("url"),
                    "playlist_index": r["metadata"].get("playlist_index"),
                    "transcript_status": r.get("transcript_status"),
                    "saved_to": r.get("saved_to"),
                }
                for r in results
            ],
        }

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)

        print(f"Summary saved to: {output_file}")
        return output_file

## Run the collector

Paste **one URL only** into `youtube_url`.

This can be any of these:
- normal YouTube video URL
- YouTube Shorts URL
- playlist URL

In [36]:
youtube_url = "https://youtu.be/aQf6Q8t1FQE?si=XY7eFyuE_2zEkOhW"

collector = UniversalYouTubeCollector(output_dir="../data/raw")

results = collector.collect_from_url(
    youtube_url=youtube_url,
    max_videos=None,          # for playlists: first 9 videos. set to None for all videos
    languages=("en",),
    parallel=True,         # used only for playlists
    max_workers=4,
)

print(f"Done. Processed {len(results)} item(s).")

Detected URL type: video
Title: An Introduction to Stress and Strain
Duration: 601 seconds
Transcript segments: 120 lang: en
Saved to: ../data/raw\video_aQf6Q8t1FQE.json
Done. Processed 1 item(s).


## Optional: save a summary file

Useful especially for playlists.

In [ ]:
# summary_path = collector.save_playlist_summary(results, filename="playlist_summary.json")
# summary_path

## Example URLs

### Normal video
```python
youtube_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
```

### Short
```python
youtube_url = "https://www.youtube.com/shorts/VIDEO_ID"
```

### Playlist
```python
youtube_url = "https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID"
```

For a full playlist, set:
```python
max_videos = None
```